# OBF-Conv

**Not a single published paper -- this repo's own combination.** Kautz/Laguerre orthonormal basis functions (OBFs) are real, well-established tools from linear system identification (Wahlberg 1991, `arXiv`-less classic; Oliveira et al. 2011 survey) for representing an impulse response compactly given a decay/resonance prior. This model transplants that idea into a conv kernel: instead of learning every spatial tap freely, the kernel is constrained to the span of a small number of fixed, orthonormal Kautz- or Laguerre-generated 2D filters, and only the combination coefficients are learned. See `model.py` for the full honesty note and the real DSP recursions used to generate the basis.

Trains on real CIFAR-10.

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from cnn_playground.data import load_cifar10
from cnn_playground.device import resolve_device
from cnn_playground.utils.seed import set_seed
from model import OBFConvModel, generate_laguerre_basis, generate_kautz_basis, _gram_schmidt

set_seed(0)
device = resolve_device('auto')
print('device:', device)

In [ ]:
# Sanity-check the generated basis is really orthonormal before training anything on top of it
lag = _gram_schmidt(generate_laguerre_basis(4, 11, pole=0.5))
kautz = _gram_schmidt(generate_kautz_basis(4, 11, r=0.75, theta=1.2))
print('laguerre max |Gram - I| =', (lag @ lag.T - torch.eye(4)).abs().max().item())
print('kautz    max |Gram - I| =', (kautz @ kautz.T - torch.eye(4)).abs().max().item())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for row in lag:
    axes[0].plot(row.numpy())
axes[0].set_title('Laguerre basis (4 sequences)')
for row in kautz:
    axes[1].plot(row.numpy())
axes[1].set_title('Kautz basis (4 sequences)')
fig.tight_layout()
plt.show()

In [ ]:
train_ds = load_cifar10(train=True)
test_ds = load_cifar10(train=False)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
print(len(train_ds), len(test_ds))

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.numel()
    return correct / total

model = OBFConvModel(basis='laguerre').to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

history = {'train_loss': [], 'test_acc': []}
epochs = 20
for epoch in range(epochs):
    model.train()
    last_loss = None
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        opt.zero_grad()
        loss = loss_fn(model(imgs), labels)
        loss.backward()
        opt.step()
        last_loss = loss.item()
    history['train_loss'].append(last_loss)
    history['test_acc'].append(evaluate(model, test_loader))

print(f"final test accuracy: {history['test_acc'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history['train_loss']); axes[0].set_title('train loss'); axes[0].set_xlabel('epoch')
axes[1].plot(history['test_acc']); axes[1].set_title('test accuracy'); axes[1].set_xlabel('epoch')
fig.tight_layout()
plt.show()

In [ ]:
classes = train_ds.classes
imgs, labels = next(iter(test_loader))
imgs, labels = imgs[:6].to(device), labels[:6]
preds = model(imgs).argmax(dim=1).cpu()

mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3,1,1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(3,1,1)

fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for i, ax in enumerate(axes):
    img = (imgs[i].cpu() * std + mean).clamp(0,1).permute(1,2,0)
    ax.imshow(img); ax.axis('off')
    ax.set_title(f'pred:{classes[preds[i]]}\ntrue:{classes[labels[i]]}', fontsize=9)
fig.tight_layout()
plt.show()